# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 02: PagedAttention & vLLM Memory Management from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In large language model serving (ChatGPT, vLLM, TGI):
- Naive systems pre-allocate a contiguous VRAM block for the KV-cache of each request up to maximum context length (e.g. 4,096 tokens).
- If a user prompt only generates 100 tokens, **97% of the allocated memory is completely wasted (Internal Fragmentation)**!
- Memory is statically reserved, limiting concurrency and leading to low GPU utilization.

In 2023, Kwon et al. (UC Berkeley) introduced **PagedAttention (vLLM)**:
- Solves KV-cache memory waste using the same principle as **Virtual Memory Paging in Operating Systems**!
- Divides KV-cache into fixed-size **Physical Blocks** (e.g. 16 tokens per block).
- Maintains a **Block Table** translating logical sequence positions to non-contiguous physical VRAM blocks.
- **Result: Slashes memory waste by 96%, enabling 2x to 4x higher serving concurrency!**

In [ ]:
import math

# Step 1: Implement PagedAttention Memory Manager from Scratch
class PagedKVCacheManager:
    def __init__(self, total_gpu_blocks=16, block_size=4):
        self.block_size = block_size
        self.total_blocks = total_gpu_blocks
        # Free list of physical block IDs
        self.free_blocks = list(range(total_gpu_blocks))
        # Logical Request ID -> list of allocated physical block IDs
        self.block_tables = {}

    def allocate_request(self, request_id, initial_tokens):
        # Calculate how many physical blocks are required
        needed_blocks = math.ceil(initial_tokens / self.block_size)
        if len(self.free_blocks) < needed_blocks:
            raise MemoryError("GPU VRAM Out of Memory! Cannot allocate new blocks.")
            
        allocated = [self.free_blocks.pop(0) for _ in range(needed_blocks)]
        self.block_tables[request_id] = allocated
        return allocated

    def append_token(self, request_id, current_total_tokens):
        # Check if the last block has room
        last_block_capacity = len(self.block_tables[request_id]) * self.block_size
        if current_total_tokens > last_block_capacity:
            # Need a new physical block on demand!
            if not self.free_blocks:
                raise MemoryError("GPU VRAM Full!")
            new_block = self.free_blocks.pop(0)
            self.block_tables[request_id].append(new_block)

    def free_request(self, request_id):
        # Reclaim physical blocks to free pool
        allocated = self.block_tables.pop(request_id, [])
        self.free_blocks.extend(allocated)

# Step 2: Simulate Multi-User Concurrent Serving
manager = PagedKVCacheManager(total_gpu_blocks=8, block_size=4)

print(f"Total Physical GPU Blocks: {manager.total_blocks} (Block size = {manager.block_size} tokens)")
print(f"Total GPU Capacity: {manager.total_blocks * manager.block_size} tokens\n")

# User 1 arrives with 7 prompt tokens
b1 = manager.allocate_request("User_1", initial_tokens=7)
print("User 1 (7 tokens) allocated physical blocks:", b1)
print("Remaining Free GPU Blocks:", manager.free_blocks)

# User 2 arrives with 5 prompt tokens
b2 = manager.allocate_request("User_2", initial_tokens=5)
print("\nUser 2 (5 tokens) allocated physical blocks:", b2)
print("Remaining Free GPU Blocks:", manager.free_blocks)

# User 1 generates token 8 and token 9 (triggers dynamic block allocation!)
manager.append_token("User_1", current_total_tokens=8)
manager.append_token("User_1", current_total_tokens=9)
print("\nUser 1 generated up to 9 tokens -> Dynamically allocated new block:")
print("User 1 Block Table:", manager.block_tables["User_1"])
print("Remaining Free GPU Blocks:", manager.free_blocks)

# User 2 finishes generation and disconnects
manager.free_request("User_2")
print("\nUser 2 completed. Freed blocks returned to pool:")
print("Free Blocks Pool:", manager.free_blocks)

### 3. Step-by-Step Code Walkthrough
1. **Zero Pre-Allocation Waste:** User 1 requested 7 tokens and received only 2 small blocks (8 slots), wasting only 1 slot instead of 4,000 slots!
2. **On-Demand Block Paging:** When generation exceeded 8 tokens, a new physical block was dynamically mapped to User 1's logical sequence.
3. **Instant Garbage Collection:** When User 2 completed, their physical memory was immediately returned to the free pool for other users.

---

### 4. Industry Applications & Trade-offs
- **vLLM Inference Engine:** Default serving backend used by Mistral, Anyscale, and major cloud LLM providers.
- **Trade-off:** Kernel complexity: GPU attention kernels must perform pointer indirection via the block table during key-value dot products.

---

### 5. Top Interview Questions & Answers
**Q1: What are the three types of memory waste in naive KV-cache serving, and how does PagedAttention eliminate them?**
> **Answer:** (1) **Reserved Waste:** Memory allocated for future tokens that are never generated. (2) **Internal Fragmentation:** Waste caused by rounding up to fixed max lengths. (3) **External Fragmentation:** Free memory scattered in non-contiguous gaps. PagedAttention eliminates all three by allocating small, non-contiguous physical pages strictly on-demand.

**Q2: How does PagedAttention enable zero-copy Parallel Sampling (Forking)?**
> **Answer:** In beam search or parallel branch generation (e.g. generating 4 responses to 1 prompt), multiple outputs share the exact same prompt tokens. PagedAttention allows multiple logical block tables to point to the **same physical prompt blocks** via reference counting (Copy-on-Write), saving 75% VRAM!